# Qwen2.5-1.5B Reasoning v4 — Colab GPU 训练流程

**适用场景**：本地无 GPU，已在本地用 `run_local.sh` 完成数据准备 / API 评测 /（可选）Teacher 数据生成。

**Colab 运行环境**：A100 (40GB) 推荐 / L4 / T4 也可（T4 自动切 fp16）。

**流程**：安装依赖 → 挂载 Drive → 克隆代码 → 同步本地产物 → **将 `outputs/` 软链到 Drive**（与 `run_gpu.sh` 内硬编码路径一致）→ 跑 `run_gpu.sh` → 同步回 Drive。

若希望 **只占用 GPU 做训练、不在 Colab 上跑慢速全量评测**：在 **Cell 7** 为 `run_gpu.sh` 加上 `--skip-eval`（见该 Cell 下方说明），将 `outputs/` 拉回本机后执行 `bash run_eval_local.sh`；本机先跑 `python3 scripts/check_eval_env.py` 自测环境。

> 断线重连：从 **Cell 3（挂载 Drive）** 开始即可，Drive 里的产物不会丢失。

In [ ]:
# Cell 1：安装依赖（每次新 Session 都要跑）
# 与仓库 requirements.txt 对齐；unsloth 会拉取兼容的 torch/transformers 组合
!pip install -q -U pip
!pip install -q "unsloth>=2025.1.0" "trl>=0.14.0" "peft>=0.14.0" "bitsandbytes>=0.45.0" \
    "transformers>=4.49.0" "datasets>=3.2.0" "accelerate>=1.2.0" \
    "pyyaml>=6.0.2" "safetensors" "tqdm" "packaging"

import unsloth, trl, transformers, peft
print(
    f"✅ Unsloth {unsloth.__version__} | TRL {trl.__version__} | "
    f"Transformers {transformers.__version__} | PEFT {peft.__version__}"
)

In [ ]:
# Cell 2：从 Colab Secrets 加载 HF_TOKEN 和 DASHSCOPE_API_KEY
# 在 Colab 左侧 🔑 Secrets 面板里添加这两个 key（名称须完全一致）
import os
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
    print("✅ HF_TOKEN 已加载")
except Exception as e:
    raise RuntimeError("请在 Colab Secrets 中添加 HF_TOKEN（HuggingFace）") from e

try:
    os.environ["DASHSCOPE_API_KEY"] = userdata.get("DASHSCOPE_API_KEY")
    print("✅ DASHSCOPE_API_KEY 已加载（Iterative DPO / 在线 Teacher 用）")
except Exception:
    print("⚠️ 未配置 DASHSCOPE_API_KEY：Iterative DPO 与在线 Teacher 不可用；SFT+DPO 训练仍可进行")

In [ ]:
# Cell 3：挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
!mkdir -p "{DRIVE_DIR}/data/processed" "{DRIVE_DIR}/outputs" "{DRIVE_DIR}/logs"
print(f"✅ Drive 已挂载，工作根目录：{DRIVE_DIR}")

In [ ]:
# Cell 4：克隆 / 更新代码（数据缓存放在 Drive，不受影响）
import os

PROJECT_DIR = "/content/6000Q-QwenMiniReason"
REPO_URL    = "https://github.com/yukiiii0730/6000Q-QwenMiniReason.git"  # ← 改成你的仓库

if not os.path.exists(PROJECT_DIR):
    print("📥 首次克隆项目...")
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("🔄 项目已存在，拉取最新代码...")
    !git -C {PROJECT_DIR} fetch origin && git -C {PROJECT_DIR} reset --hard origin/main

%cd {PROJECT_DIR}
!git log --oneline -3

In [ ]:
# Cell 5：从 Drive 同步本地阶段的产物到项目目录（data + logs）
# 本地运行 run_local.sh 后，请把 data/processed/ 和 logs/ 上传到 Drive 的 Qwen-Reasoning/ 下
DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"

!mkdir -p data/processed logs
!rsync -av --info=progress2 "{DRIVE_DIR}/data/processed/" data/processed/ || true
!rsync -av --info=progress2 "{DRIVE_DIR}/logs/" logs/ || true

print("\n📦 已同步的本地产物：")
!ls -la data/processed/ 2>/dev/null | head -20
!echo '---'
!ls -la logs/ 2>/dev/null | head -20

In [ ]:
# Cell 6：检测 GPU + 将 outputs/ 软链到 Drive（与 run_gpu.sh 中 outputs/sft、outputs/sft_merged 等路径一致）
# 注意：不要改 yaml 里的 output_dir 为绝对路径，否则 merge_lora 仍写 outputs/sft_merged 会错位。
import os
import shutil
import torch
import yaml

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb = (
    round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    if torch.cuda.is_available()
    else 0
)
print(f"GPU : {gpu_name}")
print(f"显存: {vram_gb} GB")

DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
os.makedirs(f"{DRIVE_DIR}/outputs", exist_ok=True)

out = "outputs"
if os.path.islink(out):
    print(f"✅ {out}/ 已是软链 → {os.readlink(out)}")
elif os.path.isdir(out):
    try:
        if not any(os.scandir(out)):
            shutil.rmtree(out)
            os.symlink(f"{DRIVE_DIR}/outputs", out, target_is_directory=True)
            print(f"✅ 空目录已替换为软链: {out} → {DRIVE_DIR}/outputs")
        else:
            print(f"⚠️ {out}/ 非空，未改软链；大文件仍在 /content 磁盘，注意空间")
    except OSError as e:
        print(f"⚠️ 无法创建软链: {e}")
elif not os.path.exists(out):
    os.symlink(f"{DRIVE_DIR}/outputs", out, target_is_directory=True)
    print(f"✅ 已创建软链: {out} → {DRIVE_DIR}/outputs")

# T4 / 无 bf16：写回 yaml；显存紧张时降低各 stage max_samples
for cfg_path in ("config/sft_config.yaml", "config/dpo_config.yaml"):
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    if "T4" in gpu_name or (
        torch.cuda.is_available() and not torch.cuda.is_bf16_supported()
    ):
        cfg.setdefault("train", {})["bf16"] = False
        cfg.setdefault("train", {})["fp16"] = True
        for st in cfg.get("stages", []):
            t = st.get("train")
            if t:
                t["bf16"] = False
                t["fp16"] = True
    if vram_gb and vram_gb < 20:
        for st in cfg.get("stages", []):
            ds = st.get("dataset", {})
            if ds.get("max_samples", 0) > 8000:
                ds["max_samples"] = 8000
    with open(cfg_path, "w", encoding="utf-8") as f:
        yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)

# DPO 基座必须是合并后的完整模型（与 dpo_config.yaml 一致）
with open("config/dpo_config.yaml", encoding="utf-8") as f:
    dpo = yaml.safe_load(f)
assert dpo.get("base_adapter_path") == "outputs/sft_merged", (
    "base_adapter_path 应为 outputs/sft_merged，若被改坏请 git checkout -- config/dpo_config.yaml"
)
print("✅ GPU 检测与配置补丁完成（outputs 软链 + bf16/fp16 + 可选降采样）")

In [ ]:
# Cell 7：一键运行 GPU 阶段（SFT 五段课程 → merge → 评测 → DPO → merge → 评测）
import os
import subprocess

_flags: list[str] = []
if os.path.isfile("data/processed/sft_train_filtered.json"):
    _flags.append("--use-filtered-data")
if os.path.isfile("data/processed/dpo_teacher_round_1.json"):
    _flags.append("--use-teacher-dpo")
# 只训练+合并、评测在本地：取消下一行注释
# _flags.append("--skip-eval")
print("run_gpu.sh 附加参数:", _flags or ["(无)"])

os.chmod("run_gpu.sh", 0o755)
cmd = ["bash", "run_gpu.sh", *_flags]
print("执行:", " ".join(cmd))
subprocess.run(cmd, check=True)

**Cell 7 备选用法**（在 `_flags` 上追加，或把 `cmd` 改成例如 `["bash", "run_gpu.sh", "--quick", *_flags]`）：

- `--skip-eval`：不跑 Colab 上的 GSM8K/MATH/BBH 本地评（省大量时间）；同步 `outputs/` 到本机后执行 `bash run_eval_local.sh`；本机先 `python3 scripts/check_eval_env.py`
- `--quick`：快速测试（SFT≈50 步、DPO≈30 步）
- `--skip-sft`：已有 `outputs/sft` 时跳过 SFT
- `--iterative-dpo 2`：需已配置 `DASHSCOPE_API_KEY`
- `--use-filtered-data` / `--use-teacher-dpo`：Cell 7 已按文件是否存在自动附加；也可手写进 `cmd` 列表

In [ ]:
# Cell 9：训练完成后的对比表 + 关键 JSON 指标速览
import json
import os
import subprocess

subprocess.run(["python3", "eval/compare_table.py"], check=False)

_TAGS = (
    "baseline",
    "qwen25_7b",
    "qwen25_7b_sanity",
    "qwen25_14b",
    "qwen25_14b_sanity",
    "qwen3_235b",
    "sft",
    "result",
)


def _acc_from_json(path: str) -> float | None:
    if not os.path.exists(path):
        return None
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
    if "macro_avg_accuracy" in d or "micro_avg_accuracy" in d:
        v = d.get("macro_avg_accuracy")
        if v is None:
            v = d.get("micro_avg_accuracy")
        return float(v) if v is not None else None
    if "accuracy" in d:
        return float(d["accuracy"])
    return None


summary: dict[str, float | None] = {}
for tag in _TAGS:
    for ds in ("gsm8k", "math", "bbh"):
        p = f"logs/{ds}_{tag}.json"
        acc = _acc_from_json(p)
        if acc is not None:
            summary[f"{ds}_{tag}"] = round(acc, 4)

print(json.dumps(summary, indent=2, ensure_ascii=False))

In [ ]:
# Cell 10：把训练产物 / 评测结果同步回 Drive（若 Cell 6 已软链 outputs，此处会增量同步到同一目录）
DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
!rsync -av --info=progress2 outputs/ "{DRIVE_DIR}/outputs/" 2>/dev/null || true
!rsync -av --info=progress2 logs/ "{DRIVE_DIR}/logs/" 2>/dev/null || true
print("✅ 已 rsync 到 Drive（软链场景下多为无操作或元数据更新）")

In [ ]:
# Cell 11（可选）：推理对比 — 基础模型 vs 微调模型（路径与 Cell 6 软链一致：outputs/ → Drive）
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
FT_MODEL = "outputs/merged"  # run_gpu.sh 最终合并目录
QUESTION = (
    "小明有 12 个苹果，给了小红 3 个，又买了 5 个，现在有几个？请一步步推理后给出答案。"
)


def infer(model_path: str, prompt: str, max_new: int = 512) -> str:
    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    inputs = tok(prompt, return_tensors="pt").to(mdl.device)
    outputs = mdl.generate(**inputs, max_new_tokens=max_new, do_sample=False)
    return tok.decode(outputs[0], skip_special_tokens=True)


print("=" * 60)
print("【基础模型】")
print(infer(BASE_MODEL, QUESTION))
print("\n" + "=" * 60)
print("【微调模型（SFT + DPO）】")
if not __import__("pathlib").Path(FT_MODEL).exists():
    print(f"⚠️ 未找到 {FT_MODEL}，请先完成 Cell 7 训练")
else:
    print(infer(FT_MODEL, QUESTION))